In [1]:
!pip install transformers torch pandas tqdm

  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.0 MB/s  0:00:0021.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 21.8 MB/s  0:00:031.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 25.1 MB/s  0:00:00 25.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 17.8 MB/s  0:00:00
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
  Attempting uninstall: huggingface-hub0m╸━━━━━━━━━━━━━━━━━━━━━━━  5/12 [networkx]s]
    Found existing installation: huggingface_hub 1.1.5m╸━━━━━━━━━━━━━━━━━━━━━━━  5/12 [networkx]
    Uninstalling huggingface_hub-1.1.5:38;2;249;38;114m╸━━━━━━━━━━━━━━━━━━━━━━━  5/12 [networkx]


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import os

# ==========================================
# CONFIGURATION (Colab Paths)
# ==========================================
# Input files
NEWS_FILE = "news_sentiment_inputs.csv"
REDDIT_FILE = "reddit_sentiment_inputs.csv"

# Output files
NEWS_OUTPUT = "ready_news_sentiment.csv"
REDDIT_OUTPUT = "ready_reddit_sentiment.csv"

BATCH_SIZE = 64  # Increased for T4 GPU

# ==========================================
# MODEL SETUP
# ==========================================
print("Loading FinBERT model...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

# Critical: Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on: {device.upper()} (If this says CPU, change Runtime type!)")

# ==========================================
# PROCESSING FUNCTION
# ==========================================
def process_file(input_path, output_path, text_col, id_col, ticker_col):
    if not os.path.exists(input_path):
        print(f"Skipping {input_path} (not found)")
        return

    print(f"Processing {input_path}...")
    df = pd.read_csv(input_path)
    
    # Label Mapping (ProsusAI/finbert specific)
    label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}

    # Prepare output file with header
    with open(output_path, 'w') as f:
        # Schema matches MySQL: id, ticker, label, score
        f.write(f"{id_col},ticker,sentiment_label,confidence_score\n")

    model.eval()
    total_rows = len(df)
    
    with torch.no_grad():
        for i in tqdm(range(0, total_rows, BATCH_SIZE)):
            batch = df.iloc[i : i + BATCH_SIZE]
            
            # Prepare inputs
            # Handle non-string text (rare NaN bug safety)
            texts = batch[text_col].fillna("").astype(str).tolist()
            
            # Tokenize
            inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Inference
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
            
            # Write batch to disk
            rows_to_save = []
            for j, score_tensor in enumerate(predictions):
                score = score_tensor.max().item()
                label_idx = score_tensor.argmax().item()
                label = label_map[label_idx]
                
                original_row = batch.iloc[j]
                
                # Format: ID,Ticker,Label,Score
                rows_to_save.append(f"{original_row[id_col]},{original_row[ticker_col]},{label},{score:.4f}\n")
            
            with open(output_path, 'a') as f:
                f.writelines(rows_to_save)

    print(f"Finished! Saved to {output_path}")

# ==========================================
# EXECUTION
# ==========================================
# Process News
# News CSV columns: article_id, ticker, text_input
process_file(NEWS_FILE, NEWS_OUTPUT, text_col="text_input", id_col="article_id", ticker_col="ticker")

# Process Reddit
# Reddit CSV columns: post_id, ticker, text_input
process_file(REDDIT_FILE, REDDIT_OUTPUT, text_col="text_input", id_col="post_id", ticker_col="ticker")